# 🎨 TravelMate AI - 13 Streamlit Frontend (Fixed)

This version fixes disappearing results by storing generated recommendations and itineraries in `st.session_state`.


In [1]:
# Run once in the VS Code terminal if needed:
# pip install streamlit requests streamlit-folium

print('Frontend dependencies ready ✅')


Frontend dependencies ready ✅


In [2]:
from pathlib import Path

app_code = 'import requests\nimport streamlit as st\nimport pandas as pd\nimport folium\nfrom streamlit_folium import st_folium\n\nAPI_BASE_URL = "http://127.0.0.1:8000"\n\nst.set_page_config(\n    page_title="TravelMate AI",\n    page_icon="🌍",\n    layout="wide"\n)\n\nst.title("🌍 TravelMate AI")\nst.subheader("Your personalized AI travel planner ✈️")\n\nst.write(\n    "Tell TravelMate what kind of trip you want, "\n    "and it will recommend places and build a day-by-day itinerary."\n)\n\n# -------------------------------------------------\n# Session state\n# -------------------------------------------------\n# Streamlit reruns the script whenever a widget changes.\n# The Generate button is True only for the click rerun.\n# Therefore, results must be stored in session_state.\nif "trip_result" not in st.session_state:\n    st.session_state.trip_result = None\n\nif "api_error" not in st.session_state:\n    st.session_state.api_error = None\n\n\n# -------------------------------------------------\n# Sidebar\n# -------------------------------------------------\nst.sidebar.header("🧳 Trip Preferences")\n\ndestination = st.sidebar.text_input(\n    "Destination",\n    value="Manali"\n)\n\ndays = st.sidebar.number_input(\n    "Number of days",\n    min_value=1,\n    max_value=14,\n    value=3,\n    step=1\n)\n\ntop_n = st.sidebar.slider(\n    "Number of recommendations",\n    min_value=3,\n    max_value=15,\n    value=6\n)\n\nst.sidebar.markdown("### ❤️ What do you like?")\n\nnature = st.sidebar.slider("🌿 Nature", 0.0, 1.0, 0.8, 0.1)\nhistory = st.sidebar.slider("🏛️ History", 0.0, 1.0, 0.2, 0.1)\nculture = st.sidebar.slider("🎭 Culture", 0.0, 1.0, 0.3, 0.1)\nadventure = st.sidebar.slider("🥾 Adventure", 0.0, 1.0, 0.4, 0.1)\nphotography = st.sidebar.slider("📸 Photography", 0.0, 1.0, 0.8, 0.1)\nshopping = st.sidebar.slider("🛍️ Shopping", 0.0, 1.0, 0.2, 0.1)\nreligious = st.sidebar.slider("🛕 Religious", 0.0, 1.0, 0.1, 0.1)\nfamily = st.sidebar.slider("👨\u200d👩\u200d👧 Family", 0.0, 1.0, 0.4, 0.1)\n\nquery = st.text_area(\n    "💬 Describe your trip",\n    value=(\n        "I want a peaceful scenic trip with "\n        "beautiful places for photography."\n    ),\n    height=100\n)\n\npreferences = {\n    "nature": nature,\n    "history": history,\n    "culture": culture,\n    "adventure": adventure,\n    "photography": photography,\n    "shopping": shopping,\n    "religious": religious,\n    "family": family\n}\n\n\n# -------------------------------------------------\n# Generate\n# -------------------------------------------------\ngenerate = st.button(\n    "✨ Generate My Trip",\n    type="primary",\n    use_container_width=True\n)\n\nif generate:\n    request_payload = {\n        "destination": destination,\n        "query": query,\n        "days": int(days),\n        "top_n": int(top_n),\n        "preferences": preferences\n    }\n\n    try:\n        with st.spinner("TravelMate is thinking... 🤖"):\n\n            recommendation_response = requests.post(\n                f"{API_BASE_URL}/recommend",\n                json=request_payload,\n                timeout=60\n            )\n\n            itinerary_response = requests.post(\n                f"{API_BASE_URL}/itinerary",\n                json=request_payload,\n                timeout=60\n            )\n\n        recommendation_response.raise_for_status()\n        itinerary_response.raise_for_status()\n\n        # Save successful result so it survives Streamlit reruns.\n        st.session_state.trip_result = {\n            "destination": destination,\n            "query": query,\n            "recommendations": recommendation_response.json().get(\n                "recommendations",\n                []\n            ),\n            "itinerary": itinerary_response.json().get(\n                "itinerary",\n                []\n            )\n        }\n\n        st.session_state.api_error = None\n\n    except requests.exceptions.ConnectionError:\n        st.session_state.trip_result = None\n        st.session_state.api_error = (\n            "❌ FastAPI is not running. Start it with: "\n            "`uvicorn app.main:app --reload`"\n        )\n\n    except requests.exceptions.Timeout:\n        st.session_state.trip_result = None\n        st.session_state.api_error = (\n            "⏳ The backend took too long to respond."\n        )\n\n    except requests.exceptions.RequestException as exc:\n        st.session_state.trip_result = None\n        st.session_state.api_error = f"❌ API request failed: {exc}"\n\n    except Exception as exc:\n        st.session_state.trip_result = None\n        st.session_state.api_error = f"❌ Unexpected error: {exc}"\n\n\n# -------------------------------------------------\n# Persistent output\n# -------------------------------------------------\nif st.session_state.api_error:\n    st.error(st.session_state.api_error)\n\nresult = st.session_state.trip_result\n\nif result:\n\n    st.success(\n        f"✅ Trip generated for {result[\'destination\']}"\n    )\n\n    # ---------------------------------------------\n    # Recommendations\n    # ---------------------------------------------\n    st.header("🎯 Recommended Places")\n\n    recommendations = result["recommendations"]\n\n    if recommendations:\n\n        rec_df = pd.DataFrame(recommendations)\n\n        display_columns = [\n            "name",\n            "activity_type",\n            "rating",\n            "reviews",\n            "estimated_visit_minutes",\n            "api_score"\n        ]\n\n        available = [\n            col for col in display_columns\n            if col in rec_df.columns\n        ]\n\n        st.dataframe(\n            rec_df[available],\n            use_container_width=True,\n            hide_index=True\n        )\n\n    else:\n        st.warning("No recommendations were returned.")\n\n    # ---------------------------------------------\n    # Itinerary\n    # ---------------------------------------------\n    st.header("📅 Your Itinerary")\n\n    itinerary = result["itinerary"]\n\n    if itinerary:\n\n        itinerary_df = pd.DataFrame(itinerary)\n\n        for day in sorted(itinerary_df["day"].unique()):\n\n            day_df = itinerary_df[\n                itinerary_df["day"] == day\n            ].sort_values("stop")\n\n            st.subheader(f"📍 Day {int(day)}")\n\n            for _, row in day_df.iterrows():\n\n                st.markdown(\n                    f"""\n**Stop {int(row[\'stop\'])}: {row[\'place\']}**  \n🕘 {row[\'arrival\']} → {row[\'departure\']}  \n🎯 Score: {row[\'score\']}  \n⏱️ Visit: {int(row[\'visit_minutes\'])} min  \n🚗 Travel before stop: {row[\'travel_before_minutes\']} min\n"""\n                )\n\n            st.divider()\n\n    else:\n        st.warning("No itinerary could be generated.")\n\n    # ---------------------------------------------\n    # Map\n    # ---------------------------------------------\n    st.header("🗺️ Trip Map")\n\n    valid_points = [\n        item for item in itinerary\n        if item.get("latitude") is not None\n        and item.get("longitude") is not None\n    ]\n\n    if valid_points:\n\n        center_lat = sum(\n            item["latitude"] for item in valid_points\n        ) / len(valid_points)\n\n        center_lon = sum(\n            item["longitude"] for item in valid_points\n        ) / len(valid_points)\n\n        travel_map = folium.Map(\n            location=[center_lat, center_lon],\n            zoom_start=13\n        )\n\n        for item in valid_points:\n\n            folium.Marker(\n                location=[\n                    item["latitude"],\n                    item["longitude"]\n                ],\n                popup=(\n                    f"Day {item[\'day\']} - Stop {item[\'stop\']}<br>"\n                    f"{item[\'place\']}<br>"\n                    f"{item[\'arrival\']} - {item[\'departure\']}"\n                ),\n                tooltip=(\n                    f"Day {item[\'day\']} - Stop {item[\'stop\']}: "\n                    f"{item[\'place\']}"\n                )\n            ).add_to(travel_map)\n\n        for day in sorted({\n            item["day"] for item in valid_points\n        }):\n\n            day_points = sorted(\n                [\n                    item for item in valid_points\n                    if item["day"] == day\n                ],\n                key=lambda x: x["stop"]\n            )\n\n            coordinates = [\n                [item["latitude"], item["longitude"]]\n                for item in day_points\n            ]\n\n            if len(coordinates) >= 2:\n                folium.PolyLine(\n                    coordinates,\n                    tooltip=f"Day {day} route"\n                ).add_to(travel_map)\n\n        st_folium(\n            travel_map,\n            width=None,\n            height=600\n        )\n\nelse:\n    st.info(\n        "👈 Choose your preferences and click "\n        "**Generate My Trip**."\n    )\n\nst.caption(\n    "TravelMate AI • Recommendation + Itinerary + Map"\n)\n'

app_dir = Path('../app')
app_dir.mkdir(parents=True, exist_ok=True)

app_path = app_dir / 'streamlit_app.py'
app_path.write_text(app_code, encoding='utf-8')

print(f'✅ Fixed Streamlit app created: {app_path}')


✅ Fixed Streamlit app created: ..\app\streamlit_app.py


## Why the results were disappearing

Streamlit reruns the whole script when a widget changes. A `st.button()` is only `True` during the rerun caused by that click.

The old app displayed results only inside `if generate:`. On the next rerun, `generate` became `False`, so the result disappeared.

The fixed app stores the API output in:

```python
st.session_state.trip_result
```

and renders that stored result independently of the button state.

## Run the fixed application

From the project root:

```powershell
uvicorn app.main:app --reload
```

In a second terminal:

```powershell
streamlit run app/streamlit_app.py
```
